In [2]:
import os
import pandas as pd
import emoji
from collections import Counter

# Leitura do dataset
df = pd.read_csv("../../../../00-source/TuPy/binary/tupy_binary_vote.csv")
df.head()

,text,hate
0,@user @user @user quanto vc pagava na época da...,1
1,@user os árabes já vão lhes chutar do país ??,1
2,@user @user @user @user @user tem que desenhar...,1
3,@user @user chola mais gado. e se não quiser p...,1
4,michele micheque nao tinha cartao do bolsonaro...,1


In [3]:
# Extração dos emojis dos textos
def extrair_emojis(texto):
    if pd.isna(texto):
        return []
    
    return [char for char in texto if char in emoji.EMOJI_DATA]

df["emojis"] = df["text"].apply(extrair_emojis)

# Identificação das mensagens de ódio
df_odio = df[df["hate"] == 1] # 1 = discurso de ódio
df_nao_odio = df[df["hate"] == 0] # 0 = não discurso de ódio

# Conta quantas vezes cada emoji aparece nas mensagens de ódio
emojis_odio = Counter(
    e for lista in df_odio["emojis"] for e in lista
)

# Conta quantas vezes cada emoji aparece nas mensagens de não ódio
emojis_nao_odio = Counter(
    e for lista in df_nao_odio["emojis"] for e in lista
)

# Junta todos os emojis encontrados nas duas categorias
todos = set(emojis_odio) | set(emojis_nao_odio)

# Montagem da tabela final
df_emojis = pd.DataFrame({
    "emoji": list(todos),
    "freq_odio": [emojis_odio.get(e, 0) for e in todos],
    "freq_nao_odio": [emojis_nao_odio.get(e, 0) for e in todos],
})

df_emojis["freq_total"] = (
    df_emojis["freq_odio"] + df_emojis["freq_nao_odio"]
)

# Normalização das frequências
total_odio = df_odio.shape[0]
total_nao_odio = df_nao_odio.shape[0]

df_emojis["freq_odio_norm"] = (
    df_emojis["freq_odio"] / total_odio
)

df_emojis["freq_nao_odio_norm"] = (
    df_emojis["freq_nao_odio"] / total_nao_odio
)

# Salvamento da tabela original
df_emojis.to_csv(
    "twitter_emojis_TuPy.csv",
    index=False,
    encoding="utf-8"
)

# Salvamento da tabela normalizada
df_emojis.to_csv(
    "twitter_emojis_TuPy_normalizado.csv",
    index=False,
    encoding="utf-8"
)